# Neido Studio · Imágenes de la prueba técnica
Pulsa **Entorno de ejecución → Ejecutar todo** (o Ctrl+F9). Tarda unos 15-25 minutos. No cierres la pestaña. Al final se descarga **neido-imagenes.zip**: mándaselo a Claude.

In [ ]:
# Neido Studio · Etapa "imagenes" (se ejecuta en Kaggle con GPU T4).
# 1) Genera una imagen de referencia de cada personaje (retrato de frente).
# 2) Genera el fotograma clave de cada plano; si sale un personaje, usa IP-Adapter (cara) con su referencia
#    para mantener el mismo rostro entre planos.
# Salida: /content/entrega/plano_NN.png y ref_<personaje>.png
#
# ⚠️ BORRADOR SIN PROBAR: se valida en la prueba técnica. Modelos y licencias a confirmar:
#   - Modelo base: SDXL fotorrealista (por defecto RealVisXL V5.0, derivado de SDXL; licencia a verificar)
#   - IP-Adapter Plus Face SDXL (h94/IP-Adapter, Apache 2.0)
import json, os, subprocess, sys, time
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers>=0.32", "transformers", "accelerate"], check=True)
import torch
from diffusers import StableDiffusionXLPipeline
from PIL import Image

PLANOS = json.loads('{\n  "titulo": "Prueba técnica · Titanic",\n  "rotulo": "Atlántico Norte · 14 de abril de 1912",\n  "estilo": "Cinematic film still, photorealistic, shot on 35mm with a cinema camera, natural motivated lighting, realistic skin texture with pores and imperfections, subtle film grain, slightly muted natural colours, shallow depth of field, atmospheric haze, composed for a 2.39:1 widescreen crop with the important action in the central band",\n  "negativo": "text, watermark, logo, neon, oversaturated, plastic skin, cartoon, illustration, deformed hands, extra fingers, blurry face",\n  "personajes": {\n    "santi": {\n      "descripcion": "a 32-year-old Colombian man from Medellín, light brown skin, short black hair with shaved sides, short stubble beard, brown eyes, slim build, wearing a plain yellow t-shirt, an open blue denim jacket, dark jeans, white sneakers and a plain black cap with no logos",\n      "voz": {\n        "idioma": "es",\n        "acento": "colombiano de Medellín",\n        "tono": "masculina, 30-35 años, cálida, rápida, con humor"\n      }\n    },\n    "margaret": {\n      "descripcion": "a 55-year-old British aristocratic woman in 1912, fair skin, grey hair in a high bun, angular face with marked wrinkles, grey-blue eyes, wearing a dark burgundy Edwardian evening gown, a light fur stole, a pearl necklace and long ivory gloves",\n      "voz": {\n        "idioma": "es",\n        "acento": "español formal neutro",\n        "tono": "femenina, 55 años, aguda, altiva, pausada"\n      }\n    },\n    "narrador": {\n      "voz": {\n        "idioma": "es",\n        "acento": "español neutro",\n        "tono": "masculina, grave, serena, algo misteriosa"\n      }\n    }\n  },\n  "planos": [\n    {\n      "n": 1,\n      "duracion": 8.0,\n      "tipo": "imagen_fija",\n      "movimiento": "acercamiento",\n      "personajes": [],\n      "prompt": "Wide establishing shot of the RMS Titanic sailing at night on a perfectly calm, mirror-like North Atlantic, moonless sky full of stars, warm lights glowing from the portholes and decks, faint smoke from the funnels, cold atmosphere",\n      "audio": [\n        {\n          "quien": "narrador",\n          "texto": "Catorce de abril de mil novecientos doce. Esta noche, en el barco más famoso del mundo, hay alguien que no debería estar aquí."\n        }\n      ],\n      "rotulo": true,\n      "ambiente": "mar en calma, motores lejanos"\n    },\n    {\n      "n": 2,\n      "duracion": 3.0,\n      "tipo": "video",\n      "personajes": [],\n      "prompt": "Point-of-view shot from inside a dark narrow service corridor of a 1912 ocean liner, a wooden door opening onto the first-class promenade deck at night, warm golden light from the deck lamps spilling in, a faint cool cyan glow behind the viewer, wooden deck planks and a white railing visible through the door, starry sky",\n      "movimiento_video": "the door slowly swings open and the camera moves forward through it",\n      "audio": [],\n      "ambiente": "puerta, viento"\n    },\n    {\n      "n": 3,\n      "duracion": 4.0,\n      "tipo": "video",\n      "personajes": [\n        "santi"\n      ],\n      "vertical": true,\n      "prompt": "Medium shot of {santi} stepping out onto the first-class promenade deck of the Titanic at night in 1912, looking around confused and amazed, the bow to the right of the frame, wooden deck, white railing, lifeboats hanging above, warm electric wall lamps, breath visible in the cold air, starry sky",\n      "movimiento_video": "he takes a step forward and turns his head slowly, looking around",\n      "audio": [\n        {\n          "quien": "santi",\n          "tipo": "pensamiento",\n          "texto": "Ave María… esto no es el metro de Medellín."\n        }\n      ],\n      "ambiente": "viento, mar"\n    },\n    {\n      "n": 4,\n      "duracion": 2.0,\n      "tipo": "imagen_fija",\n      "movimiento": "leve",\n      "personajes": [\n        "santi"\n      ],\n      "prompt": "Close-up insert of a man\'s hand holding a modern smartphone with a black case, the screen glowing cool blue-white and blank, against the dark wooden deck of a 1912 ocean liner at night, warm lamp light in the background",\n      "superposicion": {\n        "texto": "4:04",\n        "subtexto": "Sin servicio"\n      },\n      "audio": [],\n      "ambiente": "pitido de error suave"\n    },\n    {\n      "n": 5,\n      "duracion": 3.0,\n      "tipo": "video",\n      "personajes": [\n        "margaret"\n      ],\n      "vertical": true,\n      "prompt": "Medium shot of {margaret} standing by the white railing of the Titanic\'s first-class promenade deck at night in 1912, looking at someone off-screen to the left with disdain and curiosity, one eyebrow raised, warm electric wall lamps, starry sky, calm black sea behind her, breath visible in the cold air",\n      "movimiento_video": "she slowly looks him up and down",\n      "audio": [],\n      "ambiente": "música de cuerda lejana"\n    },\n    {\n      "n": 6,\n      "duracion": 4.0,\n      "tipo": "lipsync",\n      "personajes": [\n        "margaret"\n      ],\n      "vertical": true,\n      "prompt": "Medium close-up of {margaret} on the Titanic\'s first-class deck at night in 1912, facing the camera slightly to the left, haughty expression, warm lamp light on her face, dark starry background softly blurred, face clearly visible and well lit",\n      "movimiento_video": "she speaks with a haughty expression, subtle head movement",\n      "audio": [\n        {\n          "quien": "margaret",\n          "texto": "¿Se ha perdido, caballero? Esta cubierta es de primera clase."\n        }\n      ],\n      "ambiente": "música de cuerda lejana"\n    },\n    {\n      "n": 7,\n      "duracion": 5.0,\n      "tipo": "lipsync",\n      "personajes": [\n        "santi"\n      ],\n      "vertical": true,\n      "prompt": "Medium close-up of {santi} on the Titanic\'s first-class deck at night in 1912, facing the camera slightly to the right, nervous friendly smile, warm lamp light on one side of his face, a faint cool blue glow from below from his phone, dark starry background softly blurred, face clearly visible and well lit",\n      "movimiento_video": "he speaks with a nervous friendly smile, small gestures",\n      "audio": [\n        {\n          "quien": "santi",\n          "texto": "¿Qué pasa, pues? Tranquila, mi señora… Oiga, una preguntica: ¿este barco cómo se llama?"\n        }\n      ],\n      "ambiente": "viento"\n    },\n    {\n      "n": 8,\n      "duracion": 4.0,\n      "tipo": "video",\n      "personajes": [\n        "margaret"\n      ],\n      "prompt": "Medium shot in profile of {margaret} looking proudly out at the calm black sea from the Titanic\'s railing at night in 1912, chin raised, warm lamp light outlining her profile, starry sky",\n      "movimiento_video": "she lifts her chin proudly, almost still",\n      "audio": [\n        {\n          "quien": "margaret",\n          "texto": "El Titanic, por supuesto. El barco más seguro jamás construido."\n        }\n      ],\n      "ambiente": "mar"\n    },\n    {\n      "n": 9,\n      "duracion": 4.0,\n      "tipo": "video",\n      "personajes": [\n        "santi"\n      ],\n      "vertical": true,\n      "prompt": "Close-up of the face of {santi} on the Titanic\'s deck at night in 1912, his expression collapsing into slowly dawning horror, eyes wide, the smile disappearing, warm lamp light from the side, dark background",\n      "movimiento_video": "his smile slowly fades into horror, eyes widening",\n      "audio": [\n        {\n          "quien": "santi",\n          "tipo": "pensamiento",\n          "texto": "Ay, no. No, no, no, no…"\n        }\n      ],\n      "ambiente": "silencio",\n      "final": "negro"\n    }\n  ]\n}')
MODELO = os.environ.get("NEIDO_MODELO_IMAGEN", "SG161222/RealVisXL_V5.0")
OUT = "/content/entrega"; os.makedirs(OUT, exist_ok=True)
W, H = 1344, 768                      # ~16:9, resolución nativa de SDXL
PASOS, GUIA, SEMILLA = 30, 5.0, 404
t0 = time.time()

try:
    pipe = StableDiffusionXLPipeline.from_pretrained(MODELO, dtype=torch.float16).to("cuda")
except TypeError:
    pipe = StableDiffusionXLPipeline.from_pretrained(MODELO, torch_dtype=torch.float16).to("cuda")
# ahorro de memoria en la decodificación (el nombre de la función cambia según la versión de diffusers)
for f in (getattr(pipe, "enable_vae_slicing", None), getattr(pipe.vae, "enable_slicing", None)):
    if f:
        f(); break
estilo, negativo = PLANOS["estilo"], PLANOS["negativo"]
personajes = PLANOS["personajes"]


def generar(prompt, ip_imagen=None, semilla=SEMILLA, w=W, h=H):
    g = torch.Generator("cuda").manual_seed(semilla)
    extra = {"ip_adapter_image": ip_imagen} if ip_imagen is not None else {}
    return pipe(prompt=prompt, negative_prompt=negativo, width=w, height=h, num_inference_steps=PASOS,
                guidance_scale=GUIA, generator=g, **extra).images[0]


# 1) referencias de personaje (sin IP-Adapter)
refs = {}
for nombre, p in personajes.items():
    if "descripcion" not in p:
        continue
    img = generar(f"Close-up portrait, front view, of {p['descripcion']}, neutral expression, soft even studio light, "
                  f"plain grey background. {estilo}", w=1024, h=1024)
    img.save(f"{OUT}/ref_{nombre}.png"); refs[nombre] = img
    print(f"referencia {nombre} · {time.time()-t0:.0f} s", flush=True)

# 2) planos (con IP-Adapter de cara cuando hay personaje)
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter-plus-face_sdxl_vit-h.safetensors",
                     image_encoder_folder="models/image_encoder")
pipe.set_ip_adapter_scale(0.6)
for pl in PLANOS["planos"]:
    texto = pl["prompt"]
    for nombre, p in personajes.items():
        texto = texto.replace("{" + nombre + "}", p.get("descripcion", nombre))
    prompt = f"{texto}. {estilo}"
    caras = [refs[c] for c in pl.get("personajes", []) if c in refs]
    if caras:
        img = generar(prompt, ip_imagen=caras[0], semilla=SEMILLA + pl["n"])
    else:
        pipe.set_ip_adapter_scale(0.0)
        img = generar(prompt, ip_imagen=Image.new("RGB", (224, 224)), semilla=SEMILLA + pl["n"])
        pipe.set_ip_adapter_scale(0.6)
    img.save(f"{OUT}/plano_{pl['n']:02d}.png")
    print(f"plano {pl['n']} · {time.time()-t0:.0f} s", flush=True)

json.dump({"segundos_totales": round(time.time()-t0), "modelo": MODELO}, open(f"{OUT}/metricas_imagenes.json", "w"))
print("FIN imagenes", round(time.time()-t0), "s")


In [ ]:
import shutil
shutil.make_archive("/content/neido-imagenes", "zip", "/content/entrega")
from google.colab import files
files.download("/content/neido-imagenes.zip")
print("LISTO: se descarga neido-imagenes.zip")